# Lab | Chains in LangChain

## Outline

* LLMChain
* Sequential Chains
  * SimpleSequentialChain
  * SequentialChain
* Router Chain

In [1]:
import warnings
warnings.filterwarnings('ignore')

In [2]:
import os

from dotenv import load_dotenv, find_dotenv
_ = load_dotenv(find_dotenv())

OPENAI_API_KEY  = os.getenv('OPENAI_API_KEY')
HUGGINGFACEHUB_API_TOKEN = os.getenv('HUGGINGFACEHUB_API_TOKEN')

In [3]:
#!pip install pandas

In [4]:
import pandas as pd
df = pd.read_csv('./data/Data.csv')

In [5]:
df.head()

,Product,Review
0,Queen Size Sheet Set,I ordered a king size set. My only criticism w...
1,Waterproof Phone Pouch,"I loved the waterproof sac, although the openi..."
2,Luxury Air Mattress,This mattress had a small hole in the top of i...
3,Pillows Insert,This is the best throw pillow fillers on Amazo...
4,Milk Frother Handheld\r\n,I loved this product. But they only seem to l...


## LLMChain

In [6]:
!pip install langchain langchain_community

Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 26.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [7]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_classic.chains import LLMChain

In [8]:
# temperature=0.9 chosen for creative/descriptive product responses
llm = ChatOpenAI(temperature=0.9)

In [9]:
prompt = ChatPromptTemplate.from_template(
    "Write a concise and engaging description for the following product: {product}"
)

In [10]:

chain = LLMChain(llm=llm, prompt=prompt)

C:\Users\sisaz\AppData\Local\Temp\ipykernel_33524\546483037.py:1: LangChainDeprecationWarning: The class `LLMChain` was deprecated in LangChain 0.1.17 and will be removed in 2.0.0. Use `RunnableSequence, e.g., `prompt | llm`` instead.
  chain = LLMChain(llm=llm, prompt=prompt)


In [11]:
product = "Queen Size Sheet Set"
chain.run({"product": product})

C:\Users\sisaz\AppData\Local\Temp\ipykernel_33524\2322114793.py:2: LangChainDeprecationWarning: The method `Chain.run` was deprecated in langchain-classic 0.1.0 and will be removed in 2.0.0. Use `invoke` instead.
  chain.run({"product": product})


"Transform your bed into a luxurious haven with our Queen Size Sheet Set. Made with soft and durable materials, this set includes a fitted sheet, flat sheet, and two pillowcases to give you the ultimate sleeping experience. Upgrade your bedroom decor and indulge in a comfortable night's sleep every night."

## SimpleSequentialChain

In [12]:
from langchain_classic.chains import SimpleSequentialChain

In [13]:
llm = ChatOpenAI(temperature=0.9)

# prompt template 1
first_prompt = ChatPromptTemplate.from_template(
    "Write a concise and engaging description for the following product: {product}"
)

# Chain 1
chain_one = LLMChain(llm=llm, prompt=first_prompt)

In [14]:
# prompt template 2
second_prompt = ChatPromptTemplate.from_template(
    "Given the following product description, write a catchy marketing slogan for it:\n\n{text}"
)
# chain 2
chain_two = LLMChain(llm=llm, prompt=second_prompt)

In [15]:
overall_simple_chain = SimpleSequentialChain(chains=[chain_one, chain_two],
                                             verbose=True
                                            )

In [16]:
overall_simple_chain.run(product)



> Entering new SimpleSequentialChain chain...
Experience ultimate comfort and luxury with our Queen Size Sheet Set. Made from ultra-soft and breathable materials, these sheets are perfect for a restful night's sleep. Elevate your bedroom decor with our stylish and durable sheet set, designed to provide you with the perfect night of sleep every time. Treat yourself to the ultimate relaxation experience with our Queen Size Sheet Set.
"Sleep like royalty with our Queen Size Sheet Set - where comfort meets luxury!"

> Finished chain.


'"Sleep like royalty with our Queen Size Sheet Set - where comfort meets luxury!"'

**Repeat the above twice for different products**

**Repeat the above twice for different products**

In [17]:
overall_simple_chain.run("Waterproof Phone Pouch")



> Entering new SimpleSequentialChain chain...
Protect your phone from water damage with this waterproof phone pouch. Perfect for the beach, pool, or any outdoor adventure, this pouch ensures your phone stays dry and functional. Just slip your phone inside and enjoy peace of mind while you're on the go.
"Keep your phone afloat with our waterproof pouch - never fear the water again!"

> Finished chain.


'"Keep your phone afloat with our waterproof pouch - never fear the water again!"'

In [18]:
overall_simple_chain.run("Luxury Air Mattress")



> Entering new SimpleSequentialChain chain...
Experience ultimate comfort and relaxation with our Luxury Air Mattress. Crafted with premium materials and advanced technology, this mattress provides the perfect blend of support and softness for a truly restful night's sleep. Say goodbye to uncomfortable traditional mattresses and elevate your sleeping experience with our luxurious air mattress.
"Sleep like royalty on our Luxury Air Mattress - where comfort meets technology for the ultimate night's rest!"

> Finished chain.


'"Sleep like royalty on our Luxury Air Mattress - where comfort meets technology for the ultimate night\'s rest!"'

## SequentialChain

In [19]:
from langchain_classic.chains import SequentialChain

In [20]:
llm = ChatOpenAI(temperature=0.9)

first_prompt = ChatPromptTemplate.from_template(
    "Translate the following review to English:\n\n{Review}"
)

chain_one = LLMChain(llm=llm, prompt=first_prompt,
                     output_key="English_Review"
                    )

In [21]:
second_prompt = ChatPromptTemplate.from_template(
    "Summarize the following review in 1 sentence:\n\n{English_Review}"
)

chain_two = LLMChain(llm=llm, prompt=second_prompt,
                     output_key="summary"
                    )

In [22]:
# prompt template 3: detect the language of the original review
third_prompt = ChatPromptTemplate.from_template(
    "What language is the following review written in? Reply with only the language name.\n\n{Review}"
)
# chain 3: input= Review and output= language
chain_three = LLMChain(llm=llm, prompt=third_prompt,
                       output_key="language"
                      )

In [23]:
# prompt template 4: follow up message in the original language
fourth_prompt = ChatPromptTemplate.from_template(
    "Write a follow-up response to the following summary in {language}:\n\nSummary: {summary}"
)
chain_four = LLMChain(llm=llm, prompt=fourth_prompt,
                      output_key="followup_message"
                     )

In [24]:
# overall_chain: input= Review
# and output= English_Review, summary, followup_message
overall_chain = SequentialChain(
    chains=[chain_one, chain_two, chain_three, chain_four],
    input_variables=["Review"],
    output_variables=["English_Review", "summary", "followup_message"],
    verbose=True
)

In [25]:
review = df.Review[5]
overall_chain({"Review": review})



> Entering new SequentialChain chain...


C:\Users\sisaz\AppData\Local\Temp\ipykernel_33524\2870617583.py:2: LangChainDeprecationWarning: The method `Chain.__call__` was deprecated in langchain-classic 0.1.0 and will be removed in 2.0.0. Use `invoke` instead.
  overall_chain({"Review": review})



> Finished chain.


{'Review': "Je trouve le goût médiocre. La mousse ne tient pas, c'est bizarre. J'achète les mêmes dans le commerce et le goût est bien meilleur...\r\nVieux lot ou contrefaçon !?",
 'English_Review': "I find the taste mediocre. The foam doesn't hold up, it's weird. I buy the same ones in stores and the taste is much better... Old batch or counterfeit!?",
 'summary': 'The reviewer is disappointed with the taste and quality of the product, suggesting that it may be old or counterfeit.',
 'followup_message': "Je suis désolé d'entendre que vous avez été déçu par le goût et la qualité du produit. Il est possible qu'il soit en effet périmé ou contrefait. Je vous conseille de contacter le fabricant pour signaler votre expérience et peut-être demander un remboursement ou un échange. J'espère que vous pourrez trouver une solution satisfaisante."}

**Repeat the above twice for different products or reviews**

**Repeat the above twice for different products or reviews**

In [26]:
overall_chain({"Review": df.Review[1]})



> Entering new SequentialChain chain...

> Finished chain.


{'Review': 'I loved the waterproof sac, although the opening was made of a hard plastic. I don’t know if that would break easily. But I couldn’t turn my phone on, once it was in the pouch.',
 'English_Review': 'Me encantó la bolsa impermeable, aunque la apertura estaba hecha de plástico duro. No sé si eso se rompería fácilmente. Pero no pude encender mi teléfono una vez que estaba en la bolsa.',
 'summary': 'La bolsa impermeable fue genial, pero la apertura de plástico duro me preocupaba y no pude usar mi teléfono una vez dentro.',
 'followup_message': 'Thank you for your feedback on the waterproof bag. We appreciate your concern about the hard plastic opening and the inconvenience of not being able to use your phone once inside. We will take this into consideration for future improvements and make sure to address these issues in our next product design. If you have any other suggestions or feedback, please let us know. Your input is valuable to us!'}

In [27]:
overall_chain({"Review": df.Review[3]})



> Entering new SequentialChain chain...

> Finished chain.


{'Review': 'This is the best throw pillow fillers on Amazon. I’ve tried several others, and they’re all cheap and flat no matter how much fluffing you do. Once you toss these in the dryer after you remove them from the vacuum sealed shipping material, they fluff up great',
 'English_Review': 'Este es el mejor relleno de cojines en Amazon. He probado varios otros, y todos son baratos y planos sin importar cuánto los agites. Una vez que los pongas en la secadora después de sacarlos del material de envío sellado al vacío, se inflan muy bien.',
 'summary': 'Este relleno de cojín es el mejor en Amazon ya que se inflan bien después de ser sacados del material de envío sellado al vacío.',
 'followup_message': "Thank you for sharing your positive experience with the cushion insert from Amazon. It's great to hear that they inflate well after being removed from the vacuum-sealed packaging. It sounds like a reliable and high-quality product. I will definitely consider purchasing one for myself ba

## Router Chain

In [28]:
!pip install "protobuf>=6.31.1" tf-keras

Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 26.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [29]:
physics_template = """You are a very smart physics professor. \
You are great at answering questions about physics in a concise\
and easy to understand manner. \
When you don't know the answer to a question you admit\
that you don't know.

Here is a question:
{input}"""


math_template = """You are a very good mathematician. \
You are great at answering math questions. \
You are so good because you are able to break down \
hard problems into their component parts, 
answer the component parts, and then put them together\
to answer the broader question.

Here is a question:
{input}"""

history_template = """You are a very good historian. \
You have an excellent knowledge of and understanding of people,\
events and contexts from a range of historical periods. \
You have the ability to think, reflect, debate, discuss and \
evaluate the past. You have a respect for historical evidence\
and the ability to make use of it to support your explanations \
and judgements.

Here is a question:
{input}"""


computerscience_template = """ You are a successful computer scientist.\
You have a passion for creativity, collaboration,\
forward-thinking, confidence, strong problem-solving capabilities,\
understanding of theories and algorithms, and excellent communication \
skills. You are great at answering coding questions. \
You are so good because you know how to solve a problem by \
describing the solution in imperative steps \
that a machine can easily interpret and you know how to \
choose a solution that has a good balance between \
time complexity and space complexity. 

Here is a question:
{input}"""

biology_template = """You are an excellent biologist. \
You have a deep understanding of living organisms, \
from the molecular and cellular level to entire ecosystems. \
You are skilled at observing patterns in nature, analyzing biological data, \
and explaining complex processes like evolution, genetics, physiology, and ecology. \
You can clearly communicate how life functions and adapts, \
and you make connections between different biological concepts \
to answer challenging questions.

Here is a question:
{input}"""

In [30]:
prompt_infos = [
    {
        "name": "physics", 
        "description": "Good for answering questions about physics", 
        "prompt_template": physics_template
    },
    {
        "name": "math", 
        "description": "Good for answering math questions", 
        "prompt_template": math_template
    },
    {
        "name": "history", 
        "description": "Good for answering history questions", 
        "prompt_template": history_template
    },
    {
        "name": "computer science", 
        "description": "Good for answering computer science questions", 
        "prompt_template": computerscience_template
    },
    {
        "name": "biology",
        "description": "Good for answering biology questions",
        "prompt_template": biology_template
    }
]

In [31]:
from langchain_classic.chains.router import MultiPromptChain
from langchain_classic.chains.router.llm_router import LLMRouterChain, RouterOutputParser
from langchain_core.prompts import PromptTemplate

In [32]:
llm = ChatOpenAI(temperature=0)

In [33]:
destination_chains = {}
for p_info in prompt_infos:
    name = p_info["name"]
    prompt_template = p_info["prompt_template"]
    prompt = ChatPromptTemplate.from_template(template=prompt_template)
    chain = LLMChain(llm=llm, prompt=prompt)
    destination_chains[name] = chain  
    
destinations = [f"{p['name']}: {p['description']}" for p in prompt_infos]
destinations_str = "\n".join(destinations)

In [34]:
default_prompt = ChatPromptTemplate.from_template("{input}")
default_chain = LLMChain(llm=llm, prompt=default_prompt)

In [35]:
MULTI_PROMPT_ROUTER_TEMPLATE = """Given a raw text input to a \
language model select the model prompt best suited for the input. \
You will be given the names of the available prompts and a \
description of what the prompt is best suited for. \
You may also revise the original input if you think that revising\
it will ultimately lead to a better response from the language model.

<< FORMATTING >>
Return a markdown code snippet with a JSON object formatted to look like:
```json
{{{{
    "destination": string \ name of the prompt to use or "DEFAULT"
    "next_inputs": string \ a potentially modified version of the original input
}}}}
```

REMEMBER: "destination" MUST be one of the candidate prompt \
names specified below OR it can be "DEFAULT" if the input is not\
well suited for any of the candidate prompts.
REMEMBER: "next_inputs" can just be the original input \
if you don't think any modifications are needed.

<< CANDIDATE PROMPTS >>
{destinations}

<< INPUT >>
{{input}}

<< OUTPUT (remember to include the ```json)>>"""

In [36]:
router_template = MULTI_PROMPT_ROUTER_TEMPLATE.format(
    destinations=destinations_str
)
router_prompt = PromptTemplate(
    template=router_template,
    input_variables=["input"],
    output_parser=RouterOutputParser(),
)

router_chain = LLMRouterChain.from_llm(llm, router_prompt)

In [37]:
chain = MultiPromptChain(router_chain=router_chain, 
                         destination_chains=destination_chains, 
                         default_chain=default_chain, verbose=True
                        )

C:\Users\sisaz\AppData\Local\Temp\ipykernel_33524\3038952769.py:1: LangChainDeprecationWarning: The class `MultiPromptChain` was deprecated in LangChain 0.2.12 and will be removed in 2.0.0. Use `langchain.agents.create_agent` instead. Build routing logic with `create_agent` (e.g. with subagents or prompt-selection middleware). See https://docs.langchain.com/oss/python/langchain/agents
  chain = MultiPromptChain(router_chain=router_chain,


In [38]:
chain.run("What is black body radiation?")



> Entering new MultiPromptChain chain...
physics: {'input': 'What is black body radiation?'}
> Finished chain.


"Black body radiation refers to the electromagnetic radiation emitted by a perfect black body, which is an idealized physical body that absorbs all incident electromagnetic radiation and emits radiation at all frequencies. The radiation emitted by a black body depends only on its temperature and follows a specific distribution known as Planck's law. This type of radiation is important in understanding concepts such as thermal radiation and the behavior of objects at different temperatures."

In [39]:
chain.run("what is 2 + 2")



> Entering new MultiPromptChain chain...
math: {'input': 'what is 2 + 2'}
> Finished chain.


'The answer to 2 + 2 is 4.'

In [40]:
chain.run("Why does every cell in our body contain DNA?")



> Entering new MultiPromptChain chain...
biology: {'input': 'Why does every cell in our body contain DNA?'}
> Finished chain.


"Every cell in our body contains DNA because DNA is the genetic material that carries the instructions for the development, functioning, and reproduction of all living organisms. DNA contains the information needed to build and maintain an organism, including the proteins that make up our cells and tissues. \n\nHaving DNA in every cell ensures that each cell has the necessary genetic information to carry out its specific functions and to replicate itself accurately during cell division. This ensures that the genetic information is passed on to the next generation of cells, maintaining the integrity and continuity of the organism's genetic code.\n\nAdditionally, DNA serves as a storage system for genetic information, allowing for the transmission of traits from one generation to the next through the process of inheritance. This is essential for the survival and evolution of species, as it allows for genetic variation and adaptation to changing environments over time. \n\nIn summary, eve

**Repeat the above at least once for different inputs and chains executions - Be creative!**

**Repeat the above at least once for different inputs and chains executions - Be creative!**

In [41]:
chain.run("What caused the fall of the Roman Empire?")  # should route to history



> Entering new MultiPromptChain chain...
history: {'input': 'What caused the fall of the Roman Empire?'}
> Finished chain.


"The fall of the Roman Empire is a complex and debated topic among historians. There are several factors that are believed to have contributed to the decline and eventual fall of the empire. Some of the key factors include:\n\n1. Political instability: The Roman Empire faced frequent changes in leadership, with emperors often being assassinated or overthrown. This instability weakened the central government and made it difficult to effectively govern the vast empire.\n\n2. Economic troubles: The Roman economy struggled with issues such as inflation, high taxes, and a reliance on slave labor. The empire also faced challenges in maintaining trade routes and acquiring resources, which put a strain on the economy.\n\n3. Military defeats: The Roman Empire faced increasing pressure from external threats, such as invasions by barbarian tribes and conflicts with neighboring empires. The military was stretched thin and unable to effectively defend the empire's borders.\n\n4. Social and cultural

In [42]:
chain.run("What is the difference between a stack and a queue?")  # should route to computer science



> Entering new MultiPromptChain chain...
computer science: {'input': 'What is the difference between a stack and a queue?'}
> Finished chain.


'A stack is a data structure that follows the Last In, First Out (LIFO) principle, meaning that the last element added to the stack is the first one to be removed. This is similar to a stack of plates where you can only remove the top plate.\n\nOn the other hand, a queue is a data structure that follows the First In, First Out (FIFO) principle, meaning that the first element added to the queue is the first one to be removed. This is similar to a line of people waiting for a bus where the first person to arrive is the first one to board the bus.\n\nIn summary, the main difference between a stack and a queue is the order in which elements are added and removed.'